[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RajGajjar-01/LangChain-LangGraph/blob/main/Langgraph/01_StateGraph_Basics.ipynb)

In [ ]:
# Install dependencies if running in Google Colab
import sys
if 'google.colab' in sys.modules:
    !pip install -U langgraph

## **Basics of StateGraph in LangGraph**

<div class="premium-card">
    This notebook introduces the fundamental concepts of <b>LangGraph</b> by building a simple workflow. We demonstrate how to define a <b>State</b>, create <b>Nodes</b> for processing data, and orchestrate them using a <b>StateGraph</b>. The example calculates and categorizes BMI based on weight and height.
</div>

### **1. Imports and Environmental Setup**

We import the core LangGraph components and Python's `TypedDict` for state management.

In [1]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

### **2. Define Graph State**

The <code>BMIState</code> tracks weight, height, the calculated BMI, and its category.

In [2]:
class BMIState(TypedDict):
    weight_kg: float
    height_m: float
    bmi: float
    category: str

### **3. Define Nodes**

We define two functions: one to calculate the BMI and another to label it.

In [3]:
def calculate_bmi(state: BMIState) -> BMIState:
    weight = state['weight_kg']
    height = state['height_m']

    bmi = weight/(height ** 2)

    state['bmi'] = round(bmi, 2)

    return state

def label_bmi(state: BMIState) -> BMIState:
    bmi = state["bmi"]

    if bmi < 18.5:
        state["category"] = "underweight"
    elif 18.5 <= bmi <= 25:
        state["category"] = "Normal"
    elif 25 <= bmi < 30:
        state["category"] = "Overweight"
    else:
        state["category"] = "Obese"

    return state

### **4. Build and Compile the Graph**

We assemble the nodes and edges into a <code>StateGraph</code> and compile it into a workflow.

In [5]:
graph = StateGraph(BMIState)

graph.add_node('calculate_bmi', calculate_bmi)
graph.add_node('label_bmi', label_bmi)

graph.add_edge(START, 'calculate_bmi')
graph.add_edge("calculate_bmi", "label_bmi")
graph.add_edge("label_bmi", END)

workflow = graph.compile()

### **5. Run the Workflow**

We invoke the workflow with an initial state and print the results.

In [6]:
initial_state = {'weight_kg': 80, 'height_m': 1.73}

final_state = workflow.invoke(initial_state)

print(final_state)

{'weight_kg': 80, 'height_m': 1.73, 'bmi': 26.73, 'category': 'Overweight'}\n

### **6. Visualize the Graph**

In [7]:
from IPython.display import Image
Image(workflow.get_graph().draw_mermaid_png())